Cloud-only data dependency: this notebook expects access to OncDRS/cloud data paths and will not run in a local-only environment.

# Binary NEPC Pipeline Runner

This notebook is a thin wrapper around the actual pipeline scripts. It does not reimplement the pipeline logic.

Pipeline steps:
1. Run `shared/compile_prostate_notes.py` to build `prostate_text_data.csv`.
2. Optionally run `binary_NEPC/compile_prostate_note_bundle.py` to build a gzip note bundle.
3. Run `binary_NEPC/run_NEPC_classifier.py` to generate labels.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "binary_NEPC" else Path.cwd().resolve()
PYTHON = sys.executable


def shell_join(parts):
    return " ".join(shlex.quote(str(part)) for part in parts)


def run_command(parts, env=None):
    print(shell_join(parts))
    completed = subprocess.run(parts, cwd=REPO_ROOT, env=env, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")


DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
DEFAULT_NOTES_CSV = DEFAULT_DATA_ROOT / "prostate_text_data.csv"
DEFAULT_NOTE_BUNDLE = DEFAULT_DATA_ROOT / "LLM_NEPC_labels" / "LLM_NEPC_classifier_note_bundle.json.gz"
DEFAULT_LABELS_DIR = DEFAULT_DATA_ROOT / "LLM_NEPC_labels"
DEFAULT_COHORT_SOURCE = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/prostate_arpi_survival_cohort.csv")

In [ ]:
# Parameters
MRNS = []
MRN_FILE = '/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/mrn_lists/icd_or_vte_mrns.csv'
COHORT_SOURCE = DEFAULT_COHORT_SOURCE
NOTES_CSV_PATH = DEFAULT_NOTES_CSV
NOTE_BUNDLE_PATH = DEFAULT_NOTE_BUNDLE
OUTPUT_DIR = DEFAULT_LABELS_DIR

# Repeat entries here if you want to override the default raw OncDRS note roots.
RAW_TEXT_PATHS = []

# Classifier settings
MODEL_NAME = "gpt-4o"
MAX_WORKERS = 1
MAX_RETRIES = 6
MAX_NOTES_PER_PATIENT = 30
LIMIT_MRNS = None
OVERWRITE = False

# Step toggles
RUN_COMPILE_NOTES = False
RUN_COMPILE_BUNDLE = False
RUN_CLASSIFIER = False

In [ ]:
notes_csv_exists = Path(NOTES_CSV_PATH).exists()
note_bundle_exists = Path(NOTE_BUNDLE_PATH).exists()
labels_path = Path(OUTPUT_DIR) / "LLM_NEPC_classifier_labels.tsv"
labels_exist = labels_path.exists()

print(f"Compiled notes CSV exists: {notes_csv_exists}")
print(f"Path: {NOTES_CSV_PATH}")
print(f"Note bundle exists: {note_bundle_exists}")
print(f"Path: {NOTE_BUNDLE_PATH}")
print(f"Labels TSV exists: {labels_exist}")
print(f"Path: {labels_path}")

In [ ]:
compile_notes_cmd = [PYTHON, "shared/compile_prostate_notes.py", "--output-path", NOTES_CSV_PATH, "--cohort-source", COHORT_SOURCE]

if MRNS:
    compile_notes_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    compile_notes_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        compile_notes_cmd.extend(["--raw-text-path", raw_text_path])

print(shell_join(compile_notes_cmd))

In [ ]:
if RUN_COMPILE_NOTES:
    run_command(compile_notes_cmd)
else:
    print("Skipping compile_prostate_notes.py")

In [ ]:
compile_bundle_cmd = [PYTHON, "binary_NEPC/compile_prostate_note_bundle.py", "--output-path", NOTE_BUNDLE_PATH, "--notes-csv", NOTES_CSV_PATH]

if MRNS:
    compile_bundle_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    compile_bundle_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        compile_bundle_cmd.extend(["--raw-text-path", raw_text_path])

print(shell_join(compile_bundle_cmd))

In [ ]:
if RUN_COMPILE_BUNDLE:
    run_command(compile_bundle_cmd)
else:
    print("Skipping compile_prostate_note_bundle.py")

In [ ]:
run_classifier_cmd = [
    PYTHON,
    "binary_NEPC/run_NEPC_classifier.py",
    "--notes-csv",
    NOTES_CSV_PATH,
    "--output-dir",
    OUTPUT_DIR,
    "--model",
    MODEL_NAME,
    "--max-workers",
    str(MAX_WORKERS),
    "--max-retries",
    str(MAX_RETRIES),
    "--max-notes-per-patient",
    str(MAX_NOTES_PER_PATIENT),
]

if Path(NOTE_BUNDLE_PATH).exists():
    run_classifier_cmd.extend(["--note-bundle-path", NOTE_BUNDLE_PATH])
if MRNS:
    run_classifier_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    run_classifier_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        run_classifier_cmd.extend(["--raw-text-path", raw_text_path])
if LIMIT_MRNS is not None:
    run_classifier_cmd.extend(["--limit-mrns", str(LIMIT_MRNS)])
if OVERWRITE:
    run_classifier_cmd.append("--overwrite")

print(shell_join(run_classifier_cmd))

In [ ]:
if RUN_CLASSIFIER:
    run_command(run_classifier_cmd)
else:
    print("Skipping run_NEPC_classifier.py")